In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType

spark = SparkSession.builder.appName("bronze").master("local[*]").getOrCreate()

RAW = "/home/work/data/raw/"

# Colonnes de chaque table, dans l'ordre du CSV
COLUMNS = {
    "customers": ["customer_id", "customer_unique_id", "customer_zip_code_prefix",
                  "customer_city", "customer_state"],
    "geolocation": ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng",
                    "geolocation_city", "geolocation_state"],
    "order_items": ["order_id", "order_item_id", "product_id", "seller_id",
                    "shipping_limit_date", "price", "freight_value"],
    "order_payments": ["order_id", "payment_sequential", "payment_type",
                       "payment_installments", "payment_value"],
    "order_reviews": ["review_id", "order_id", "review_score", "review_comment_title",
                      "review_comment_message", "review_creation_date",
                      "review_answer_timestamp"],
    "orders": ["order_id", "customer_id", "order_status", "order_purchase_timestamp",
               "order_approved_at", "order_delivered_carrier_date",
               "order_delivered_customer_date", "order_estimated_delivery_date"],
    "products": ["product_id", "product_category_name", "product_name_lenght",
                 "product_description_lenght", "product_photos_qty", "product_weight_g",
                 "product_length_cm", "product_height_cm", "product_width_cm"],
    "sellers": ["seller_id", "seller_zip_code_prefix", "seller_city", "seller_state"],
    "category_translation": ["product_category_name", "product_category_name_english"],
}

# Nom du fichier CSV de chaque table
FILES = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

# Schémas explicites : toutes les colonnes en string
SCHEMAS = {
    nom: StructType([StructField(c, StringType(), True) for c in cols])
    for nom, cols in COLUMNS.items()
}

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/08 20:46:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
for nom, fichier in FILES.items():
    # Pour order_reviews, on lit l'en-tête avec les mêmes options que pour les données
    opts = {"multiLine": "true", "quote": '"', "escape": '"'} if nom == "order_reviews" else {}
    entete = spark.read.options(**opts).option("header", True).csv(RAW + fichier).columns
    ok = entete == COLUMNS[nom]
    print(f"{nom:22} {'OK' if ok else 'DIFFÉRENT'}")
    if not ok:
        print("   CSV     :", entete)
        print("   Déclaré :", COLUMNS[nom])

customers              OK
geolocation            OK
order_items            OK
order_payments         OK
order_reviews          OK
orders                 OK
products               OK
sellers                OK
category_translation   OK


In [3]:
from datetime import datetime

BRONZE = "/home/work/data/bronze/"
BATCH_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

def lire_csv(nom):
    """Lit un CSV brut avec son schéma explicite, sans aucune transformation."""
    lecteur = spark.read.schema(SCHEMAS[nom]).option("header", True)
    if nom == "order_reviews":
        # Texte libre : retours à la ligne et guillemets dans les commentaires
        lecteur = lecteur.option("multiLine", True).option("quote", '"').option("escape", '"')
    return lecteur.csv(RAW + FILES[nom])

def ingerer(nom):
    """Lit un CSV, ajoute les métadonnées d'audit, écrit en Parquet dans Bronze."""
    df = (lire_csv(nom)
          .withColumn("ingestion_ts", F.current_timestamp())
          .withColumn("source_file", F.lit(FILES[nom]))
          .withColumn("batch_id", F.lit(BATCH_ID)))
    df.write.mode("overwrite").parquet(BRONZE + nom)
    return nom

for nom in FILES:
    ingerer(nom)
    print("OK :", nom)

print("Batch :", BATCH_ID)

OK : customers


26/11/08 20:51:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

OK : geolocation
OK : order_items
OK : order_payments


OK : order_reviews


OK : orders


OK : products
OK : sellers
OK : category_translation
Batch : 20261108_205133


In [4]:
for nom in FILES:
    df = spark.read.parquet(BRONZE + nom)
    print(f"{nom:22} lignes={df.count():>8}  colonnes={len(df.columns)}")

customers              lignes=   99441  colonnes=8
geolocation            lignes= 1000163  colonnes=8
order_items            lignes=  112650  colonnes=10
order_payments         lignes=  103886  colonnes=8
order_reviews          lignes=   99224  colonnes=10
orders                 lignes=   99441  colonnes=11
products               lignes=   32951  colonnes=12
sellers                lignes=    3095  colonnes=7
category_translation   lignes=      71  colonnes=5


In [5]:
# Comptages de référence : ceux du Test A, documentés dans schema.md
ATTENDU = {
    "customers": 99441, "geolocation": 1000163, "order_items": 112650,
    "order_payments": 103886, "order_reviews": 99224, "orders": 99441,
    "products": 32951, "sellers": 3095, "category_translation": 71,
}
META = ["ingestion_ts", "source_file", "batch_id"]

print(f"{'table':22} {'attendu':>9} {'bronze':>9} {'src-brz':>8} {'brz-src':>8} {'meta':>6}")
for nom in FILES:
    src = lire_csv(nom)
    brz = spark.read.parquet(BRONZE + nom)
    brz_data = brz.drop(*META)

    n_brz = brz.count()
    diff_a = src.exceptAll(brz_data).count()      # lignes de la source absentes du Bronze
    diff_b = brz_data.exceptAll(src).count()      # lignes du Bronze absentes de la source

    meta_ok = (brz.filter(" OR ".join(f"{c} IS NULL" for c in META)).count() == 0
               and brz.select("batch_id").distinct().count() == 1)

    statut = "OK" if (n_brz == ATTENDU[nom] and diff_a == 0 and diff_b == 0 and meta_ok) else "ERREUR"
    print(f"{nom:22} {ATTENDU[nom]:>9} {n_brz:>9} {diff_a:>8} {diff_b:>8} {statut:>6}")

# Contrôle de lecture sur order_reviews : les identifiants doivent faire 32 caractères
rev = spark.read.parquet(BRONZE + "order_reviews")
print("review_id de longueur != 32 :", rev.filter(F.length("review_id") != 32).count())
print("order_id de longueur != 32  :", rev.filter(F.length("order_id") != 32).count())

table                    attendu    bronze  src-brz  brz-src   meta


customers                  99441     99441        0        0     OK


geolocation              1000163   1000163        0        0     OK


order_items               112650    112650        0        0     OK
order_payments            103886    103886        0        0     OK


order_reviews              99224     99224        0        0     OK


orders                     99441     99441        0        0     OK
products                   32951     32951        0        0     OK
sellers                     3095      3095        0        0     OK
category_translation          71        71        0        0     OK
review_id de longueur != 32 : 0
order_id de longueur != 32  : 0


In [6]:
# Clé censée identifier une ligne (voir schema.md). geolocation n'en a pas.
KEYS = {
    "customers": ["customer_id"],
    "geolocation": None,
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id", "order_id"],
    "orders": ["order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "category_translation": ["product_category_name"],
}

resultats = {}   # on garde les chiffres pour le rapport (1.5)

for nom in FILES:
    brz = spark.read.parquet(BRONZE + nom)
    cols = [c for c in brz.columns if c not in META]
    d = brz.select(cols)
    total = d.count()

    # Valeurs manquantes par colonne, en un seul passage
    ligne = d.agg(*[
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
        for c in cols
    ]).first()
    manquants = {c: ligne[c] for c in cols if ligne[c]}

    # Doublons sur la clé et doublons exacts
    cle = KEYS[nom]
    dup_cle = total - d.select(*cle).distinct().count() if cle else None
    dup_exact = total - d.distinct().count()

    resultats[nom] = {"total": total, "manquants": manquants,
                      "dup_cle": dup_cle, "dup_exact": dup_exact}

    print(f"\n=== {nom} ({total:,} lignes) ===".replace(",", " "))
    print("Doublons sur la clé :", dup_cle, "| doublons exacts :", dup_exact)
    if manquants:
        for c, n in manquants.items():
            print(f"  {c:35} {n:>8}  ({n / total * 100:.1f} %)")
    else:
        print("  aucune valeur manquante")


=== customers (99 441 lignes) ===
Doublons sur la clé : 0 | doublons exacts : 0
  aucune valeur manquante



=== geolocation (1 000 163 lignes) ===
Doublons sur la clé : None | doublons exacts : 261831
  aucune valeur manquante

=== order_items (112 650 lignes) ===
Doublons sur la clé : 0 | doublons exacts : 0
  aucune valeur manquante

=== order_payments (103 886 lignes) ===
Doublons sur la clé : 0 | doublons exacts : 0
  aucune valeur manquante

=== order_reviews (99 224 lignes) ===
Doublons sur la clé : 0 | doublons exacts : 0
  review_comment_title                   87658  (88.3 %)
  review_comment_message                 58256  (58.7 %)

=== orders (99 441 lignes) ===
Doublons sur la clé : 0 | doublons exacts : 0
  order_approved_at                        160  (0.2 %)
  order_delivered_carrier_date            1783  (1.8 %)
  order_delivered_customer_date           2965  (3.0 %)

=== products (32 951 lignes) ===
Doublons sur la clé : 0 | doublons exacts : 0
  product_category_name                    610  (1.9 %)
  product_name_lenght                      610  (1.9 %)
  product_descriptio

In [7]:
vide = lambda c: F.col(c).isNull() | (F.trim(F.col(c)) == "")

# 1. Les dates manquantes de orders, selon le statut
orders = spark.read.parquet(BRONZE + "orders")
(orders.groupBy("order_status").agg(
    F.count("*").alias("total"),
    F.sum(F.when(vide("order_approved_at"), 1).otherwise(0)).alias("sans_approbation"),
    F.sum(F.when(vide("order_delivered_carrier_date"), 1).otherwise(0)).alias("sans_date_transporteur"),
    F.sum(F.when(vide("order_delivered_customer_date"), 1).otherwise(0)).alias("sans_date_livraison"))
 .orderBy(F.desc("total")).show())

# 2. Les 610 produits sont-ils les mêmes lignes ?
products = spark.read.parquet(BRONZE + "products")
print("Sans catégorie :", products.filter(vide("product_category_name")).count())
print("Sans catégorie ET sans nom, description, photos :",
      products.filter(vide("product_category_name") & vide("product_name_lenght")
                      & vide("product_description_lenght") & vide("product_photos_qty")).count())

+------------+-----+----------------+----------------------+-------------------+
|order_status|total|sans_approbation|sans_date_transporteur|sans_date_livraison|
+------------+-----+----------------+----------------------+-------------------+
|   delivered|96478|              14|                     2|                  8|
|     shipped| 1107|               0|                     0|               1107|
|    canceled|  625|             141|                   550|                619|
| unavailable|  609|               0|                   609|                609|
|    invoiced|  314|               0|                   314|                314|
|  processing|  301|               0|                   301|                301|
|     created|    5|               5|                     5|                  5|
|    approved|    2|               0|                     2|                  2|
+------------+-----+----------------+----------------------+-------------------+

Sans catégorie : 610
Sans c

In [8]:
vide = lambda c: F.col(c).isNull() | (F.trim(F.col(c)) == "")
orders = spark.read.parquet(BRONZE + "orders")

anomalies = orders.filter((F.col("order_status") == "delivered") &
    (vide("order_approved_at") | vide("order_delivered_carrier_date") | vide("order_delivered_customer_date")))
print("Commandes delivered avec au moins une date manquante :", anomalies.count())

print("Dernière commande du dataset :")
orders.agg(F.max("order_purchase_timestamp")).show(truncate=False)
print("Commandes shipped sans livraison, première et dernière date d'achat :")
(orders.filter("order_status = 'shipped'")
       .agg(F.min("order_purchase_timestamp"), F.max("order_purchase_timestamp"))
       .show(truncate=False))

Commandes delivered avec au moins une date manquante : 23
Dernière commande du dataset :
+-----------------------------+
|max(order_purchase_timestamp)|
+-----------------------------+
|2018-10-17 17:30:18          |
+-----------------------------+

Commandes shipped sans livraison, première et dernière date d'achat :
+-----------------------------+-----------------------------+
|min(order_purchase_timestamp)|max(order_purchase_timestamp)|
+-----------------------------+-----------------------------+
|2016-09-04 21:15:19          |2018-09-03 09:06:57          |
+-----------------------------+-----------------------------+



In [9]:
CONTROLES = {
    "orders": {"order_purchase_timestamp": "timestamp", "order_approved_at": "timestamp",
               "order_delivered_carrier_date": "timestamp",
               "order_delivered_customer_date": "timestamp",
               "order_estimated_delivery_date": "timestamp"},
    "order_items": {"order_item_id": "int", "shipping_limit_date": "timestamp",
                    "price": "double", "freight_value": "double"},
    "order_payments": {"payment_sequential": "int", "payment_installments": "int",
                       "payment_value": "double"},
    "order_reviews": {"review_score": "int", "review_creation_date": "timestamp",
                      "review_answer_timestamp": "timestamp"},
    "products": {"product_name_lenght": "int", "product_description_lenght": "int",
                 "product_photos_qty": "int", "product_weight_g": "double",
                 "product_length_cm": "double", "product_height_cm": "double",
                 "product_width_cm": "double"},
    "geolocation": {"geolocation_lat": "double", "geolocation_lng": "double"},
}

print(f"{'table':16} {'colonne':32} {'type':10} {'invalides':>9}")
for nom, cols in CONTROLES.items():
    df = spark.read.parquet(BRONZE + nom)
    ligne = df.agg(*[
        F.sum(F.when(~vide(c) & F.expr(f"try_cast({c} AS {t})").isNull(), 1).otherwise(0)).alias(c)
        for c, t in cols.items()
    ]).first()
    for c, t in cols.items():
        print(f"{nom:16} {c:32} {t:10} {ligne[c]:>9}")

table            colonne                          type       invalides
orders           order_purchase_timestamp         timestamp          0
orders           order_approved_at                timestamp          0
orders           order_delivered_carrier_date     timestamp          0
orders           order_delivered_customer_date    timestamp          0
orders           order_estimated_delivery_date    timestamp          0
order_items      order_item_id                    int                0
order_items      shipping_limit_date              timestamp          0
order_items      price                            double             0
order_items      freight_value                    double             0
order_payments   payment_sequential               int                0
order_payments   payment_installments             int                0
order_payments   payment_value                    double             0
order_reviews    review_score                     int                0
order_

In [10]:
def plage(nom, col, t="double"):
    v = (spark.read.parquet(BRONZE + nom)
         .select(F.expr(f"try_cast({col} AS {t})").alias("v")).filter("v IS NOT NULL"))
    r = v.agg(F.min("v"), F.max("v"), F.sum(F.when(F.col("v") <= 0, 1).otherwise(0))).first()
    print(f"{nom}.{col:28} min={r[0]}  max={r[1]}  valeurs<=0={r[2]}")

for nom, col in [("order_items", "price"), ("order_items", "freight_value"),
                 ("order_payments", "payment_value"), ("order_payments", "payment_installments"),
                 ("products", "product_weight_g"), ("products", "product_photos_qty")]:
    plage(nom, col)

print("\nPériode des commandes (achat) :")
spark.read.parquet(BRONZE + "orders").agg(
    F.min("order_purchase_timestamp"), F.max("order_purchase_timestamp")).show(truncate=False)

for nom, col in [("order_reviews", "review_score"), ("order_payments", "payment_type"),
                 ("orders", "order_status")]:
    print(f"\n{nom}.{col}")
    spark.read.parquet(BRONZE + nom).groupBy(col).count().orderBy(F.desc("count")).show(truncate=False)

# Coordonnées hors d'un rectangle englobant le Brésil (approximatif)
geo = spark.read.parquet(BRONZE + "geolocation").select(
    F.expr("try_cast(geolocation_lat AS double)").alias("lat"),
    F.expr("try_cast(geolocation_lng AS double)").alias("lng"))
print("Points hors Brésil (lat -34..6, lng -74..-34) :",
      geo.filter(~F.col("lat").between(-34, 6) | ~F.col("lng").between(-74, -34)).count())

order_items.price                        min=0.85  max=6735.0  valeurs<=0=0
order_items.freight_value                min=0.0  max=409.68  valeurs<=0=383
order_payments.payment_value                min=0.0  max=13664.08  valeurs<=0=9
order_payments.payment_installments         min=0.0  max=24.0  valeurs<=0=2
products.product_weight_g             min=0.0  max=40425.0  valeurs<=0=4
products.product_photos_qty           min=1.0  max=20.0  valeurs<=0=0

Période des commandes (achat) :
+-----------------------------+-----------------------------+
|min(order_purchase_timestamp)|max(order_purchase_timestamp)|
+-----------------------------+-----------------------------+
|2016-09-04 21:15:19          |2018-10-17 17:30:18          |
+-----------------------------+-----------------------------+


order_reviews.review_score
+------------+-----+
|review_score|count|
+------------+-----+
|5           |57328|
|4           |19142|
|1           |11424|
|3           |8179 |
|2           |3151 |
+-------

[Stage 537:=======>                                                 (1 + 7) / 8]

Points hors Brésil (lat -34..6, lng -74..-34) : 42


In [11]:
orders = spark.read.parquet(BRONZE + "orders")
pay = spark.read.parquet(BRONZE + "order_payments")
products = spark.read.parquet(BRONZE + "products")

# 1. Les paiements anormaux, avec le statut de leur commande
(pay.filter((F.expr("try_cast(payment_value AS double)") <= 0)
            | (F.expr("try_cast(payment_installments AS int)") <= 0)
            | (F.col("payment_type") == "not_defined"))
    .join(orders.select("order_id", "order_status"), "order_id", "left")
    .select("order_id", "payment_type", "payment_installments", "payment_value", "order_status")
    .show(20, truncate=False))

# 2. Les 4 produits à poids 0
(products.filter("try_cast(product_weight_g AS double) = 0")
         .select("product_id", "product_category_name", "product_weight_g").show(truncate=False))

# 3. Volume de commandes par mois (la période est-elle exploitable partout ?)
(orders.groupBy(F.date_format(F.to_timestamp("order_purchase_timestamp"), "yyyy-MM").alias("mois"))
       .count().orderBy("mois").show(30))

+--------------------------------+------------+--------------------+-------------+------------+
|order_id                        |payment_type|payment_installments|payment_value|order_status|
+--------------------------------+------------+--------------------+-------------+------------+
|8bcbe01d44d147f901cd3192671144db|voucher     |1                   |0.00         |delivered   |
|fa65dad1b0e818e3ccc5cb0e39231352|voucher     |1                   |0.00         |shipped     |
|6ccb433e00daae1283ccc956189c82ae|voucher     |1                   |0.00         |delivered   |
|744bade1fcf9ff3f31d860ace076d422|credit_card |0                   |58.69        |delivered   |
|4637ca194b6387e2d538dc89b124b0ee|not_defined |1                   |0.00         |canceled    |
|00b1cb0320190ca0daa2c88b35206009|not_defined |1                   |0.00         |canceled    |
|45ed6e85398a87c253db47c2d9f48216|voucher     |1                   |0.00         |delivered   |
|fa65dad1b0e818e3ccc5cb0e39231352|vouche

In [12]:
def lire(nom): return spark.read.parquet(BRONZE + nom)
o, c, it, pay, rev, prod, sel, geo, tr = [lire(n) for n in
    ["orders", "customers", "order_items", "order_payments", "order_reviews",
     "products", "sellers", "geolocation", "category_translation"]]

vide = lambda col: F.col(col).isNull() | (F.trim(F.col(col)) == "")

def orphelins(libelle, enfant, cle_e, parent, cle_p):
    """Valeurs non vides de cle_e (enfant) absentes de cle_p (parent)."""
    e = enfant.filter(~vide(cle_e))
    p = parent.select(F.col(cle_p).alias("_k")).distinct()
    n = e.join(p, e[cle_e] == F.col("_k"), "left_anti").count()
    print(f"{libelle:50} {n:>8}")

print("--- Enfants sans parent (intégrité) ---")
orphelins("orders.customer_id -> customers", o, "customer_id", c, "customer_id")
orphelins("order_items.order_id -> orders", it, "order_id", o, "order_id")
orphelins("order_items.product_id -> products", it, "product_id", prod, "product_id")
orphelins("order_items.seller_id -> sellers", it, "seller_id", sel, "seller_id")
orphelins("order_payments.order_id -> orders", pay, "order_id", o, "order_id")
orphelins("order_reviews.order_id -> orders", rev, "order_id", o, "order_id")
orphelins("products.category -> category_translation", prod, "product_category_name", tr, "product_category_name")
orphelins("customers.zip -> geolocation", c, "customer_zip_code_prefix", geo, "geolocation_zip_code_prefix")
orphelins("sellers.zip -> geolocation", sel, "seller_zip_code_prefix", geo, "geolocation_zip_code_prefix")

print("\n--- Parents sans enfant (information métier) ---")
orphelins("orders sans ligne produit (attendu : 775)", o, "order_id", it, "order_id")
orphelins("orders sans paiement", o, "order_id", pay, "order_id")
orphelins("orders sans avis", o, "order_id", rev, "order_id")
orphelins("customers sans commande", c, "customer_id", o, "customer_id")
orphelins("products jamais vendus", prod, "product_id", it, "product_id")
orphelins("sellers sans vente", sel, "seller_id", it, "seller_id")

--- Enfants sans parent (intégrité) ---
orders.customer_id -> customers                           0
order_items.order_id -> orders                            0
order_items.product_id -> products                        0
order_items.seller_id -> sellers                          0
order_payments.order_id -> orders                         0
order_reviews.order_id -> orders                          0
products.category -> category_translation                13
customers.zip -> geolocation                            278
sellers.zip -> geolocation                                7

--- Parents sans enfant (information métier) ---
orders sans ligne produit (attendu : 775)               775
orders sans paiement                                      1
orders sans avis                                        768
customers sans commande                                   0
products jamais vendus                                    0
sellers sans vente                                        0


In [13]:
ts = lambda col: F.expr(f"try_cast({col} AS timestamp)")

od = o.select("order_id", "order_status",
    ts("order_purchase_timestamp").alias("achat"),
    ts("order_approved_at").alias("approbation"),
    ts("order_delivered_carrier_date").alias("transporteur"),
    ts("order_delivered_customer_date").alias("livraison"),
    ts("order_estimated_delivery_date").alias("estimee"))

regles = {
    "approbation avant achat":        F.col("approbation") < F.col("achat"),
    "transporteur avant approbation": F.col("transporteur") < F.col("approbation"),
    "livraison avant transporteur":   F.col("livraison") < F.col("transporteur"),
    "livraison avant achat":          F.col("livraison") < F.col("achat"),
    "date estimée avant achat":       F.col("estimee") < F.col("achat"),
}
for lib, cond in regles.items():
    print(f"{lib:35} {od.filter(cond).count():>8}")

# Avis créé avant l'achat de la commande
r = (rev.select("order_id", ts("review_creation_date").alias("avis"))
        .join(od.select("order_id", "achat"), "order_id"))
print(f"{'avis créé avant l achat':35} {r.filter(F.col('avis') < F.col('achat')).count():>8}")

approbation avant achat                    0
transporteur avant approbation          1359
livraison avant transporteur              23
livraison avant achat                      0
date estimée avant achat                   0
avis créé avant l achat                   74


In [14]:
# 1. Avis créé AVANT le jour d'achat (comparaison au niveau du jour)
r = (rev.select("order_id", F.to_date(ts("review_creation_date")).alias("jour_avis"))
        .join(od.select("order_id", F.to_date("achat").alias("jour_achat")), "order_id"))
print("Avis créé un jour avant le jour d'achat :", r.filter("jour_avis < jour_achat").count())

# 2. Quelles catégories n'ont pas de traduction ?
(prod.filter(~vide("product_category_name"))
     .join(tr, "product_category_name", "left_anti")
     .groupBy("product_category_name").count().show(truncate=False))

# 3. Ampleur des écarts "transporteur avant approbation" (en heures)
(od.filter(F.col("transporteur") < F.col("approbation"))
   .select(((F.unix_timestamp("approbation") - F.unix_timestamp("transporteur")) / 3600).alias("heures_ecart"))
   .agg(F.min("heures_ecart"), F.expr("percentile_approx(heures_ecart, 0.5)"), F.max("heures_ecart"))
   .show())

Avis créé un jour avant le jour d'achat : 64
+---------------------------------------------+-----+
|product_category_name                        |count|
+---------------------------------------------+-----+
|pc_gamer                                     |3    |
|portateis_cozinha_e_preparadores_de_alimentos|10   |
+---------------------------------------------+-----+

+--------------------+-------------------------------------------+-----------------+
|   min(heures_ecart)|percentile_approx(heures_ecart, 0.5, 10000)|max(heures_ecart)|
+--------------------+-------------------------------------------+-----------------+
|0.005833333333333334|                          17.16777777777778|4109.256111111111|
+--------------------+-------------------------------------------+-----------------+

